# SpaceX Falcon 9 - Web Scraping
This notebook demonstrates the capstone web-scraping workflow using `requests`, `BeautifulSoup`, and Pandas to collect historical Falcon 9 launch records from Wikipedia.


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = 'https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches'
html = requests.get(url).text
soup = BeautifulSoup(html, 'html.parser')
print(soup.title)


## Identify launch tables
Wikipedia tables are inspected for headers such as flight number, date/time, launch site, payload, orbit and customer. The rows are then converted into a structured list.


In [ ]:
tables = soup.find_all('table', class_='wikitable')
rows = []
for table in tables:
    for tr in table.find_all('tr')[1:]:
        cells = [c.get_text(' ', strip=True) for c in tr.find_all(['th','td'])]
        if len(cells) >= 6:
            rows.append(cells[:8])

columns = ['Flight No.','Date and time','Version Booster','Launch site','Payload','Payload mass','Orbit','Customer']
scraped = pd.DataFrame(rows, columns=columns)
scraped.head()


## Cleaning
Footnote markers and non-breaking spaces are removed, while the raw strings are retained for traceability. This scraped source complements the API data and demonstrates a second collection method.


In [ ]:
for col in scraped.columns:
    scraped[col] = (scraped[col].astype(str)
                    .str.replace('\xa0',' ', regex=False)
                    .str.replace(r'\[[^\]]+\]','', regex=True)
                    .str.strip())

scraped.to_csv('spacex_web_scraped.csv', index=False)
print('Rows collected:', len(scraped))
